In [1]:
from deep_orderbook.learn.test_learn import train_and_predict
from deep_orderbook.visu import Visualizer
from deep_orderbook.config import TrainConfig, ReplayConfig, ShaperConfig
from deep_orderbook.strategy import Strategy
train_config = TrainConfig(
    num_workers=5,    batch_size=8,    data_queue_size=1024,    num_levels=8,    learning_rate=0.001,    epochs=100, save_checkpoint_batches=10, save_checkpoint_mins=1.0
)
replay_config = ReplayConfig(
    markets=["ETH-USD"],#, "BTC-USD", "ETH-BTC"],
    date_regexp='2025-02-[0,1]2T15*',
    # date_regexp='2024-11-06T20*',
    data_dir='/media/photoDS216/crypto/',
    every="100ms",
)
shaper_config = ShaperConfig(
    only_full_arrays=True,
    view_bips = 5,
    num_side_lvl = 8,
    look_ahead = 32,
    look_ahead_side_bips = 5,
    look_ahead_side_width = 4,
    rolling_window_size = 1024*8,
    window_stride = 8,
    randomize=True,
)
test_config = replay_config.but(
    # date_regexp='2025-02-2*',
    # date_regexp='2024-11-06*',
    # window_stride = 8,
    # randomize=False
)

In [ ]:
vis = Visualizer()
strategy = Strategy(threshold=0.2)
strategy2 = Strategy(threshold=0.04)
async def online_learning(train_config, replay_config, shaper_config):
    async for books_arr, t2l, pxar, pred_t2l, train_loss, test_loss in train_and_predict(
        resume_from_checkpoint=False,
        config=train_config, replay_config=replay_config, shaper_config=shaper_config, test_config=test_config
    ):
        gt_pnl, pos, gt_up_prox, gt_down_prox = strategy.compute_pnl(pxar, t2l)
        pred_pnl, pred_pos, pred_up_prox, pred_down_prox = strategy2.compute_pnl(pxar, pred_t2l)

        vis.add_loss(train_loss, test_loss)
        vis.update(
            books_z_data=books_arr, level_reach_z_data=t2l, bidask=pxar, 
            pred_t2l=pred_t2l, gt_pnl=gt_pnl, pred_pnl=pred_pnl, 
            positions=pos, pred_positions=pred_pos,
            up_proximity=gt_up_prox, down_proximity=gt_down_prox,
            pred_up_proximity=pred_up_prox, pred_down_proximity=pred_down_prox
        )
await online_learning(train_config=train_config, replay_config=replay_config, shaper_config=shaper_config)

FigureWidget({
    'data': [{'line': {'color': 'green'},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'uid': '780a1cd3-7010-4db9-a698-664bc5d03099',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'color': 'red'},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'uid': '4db035e6-e6f8-4721-b6b8-ebc626424f48',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'marker': {'color': 'green', 'size': 12, 'symbol': 'triangle-up'},
              'mode': 'markers',
              'name': 'Omniscient Entry',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4a6259c4-f49c-494c-8bce-db00e0044e0f',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxi

self.device=device(type='cuda')


2026-05-08 19:41:29.383 | DEBUG    | deep_orderbook.cache_manager:load_cached:123 - Cache hit: 34973 ticks from cache/ETHUSD_100ms_vb05_lv08_la032_lasb05_lalv04/2025-02/2025-02-02T15-00-11.npz
